In [25]:
import pandas as pd
import plotly.express as px
import numpy as np


In [5]:
tasks = pd.read_csv("submissions_archive/user_task_submissions.csv", sep=";")
practices = pd.read_csv("submissions_archive/user_practice_submissions.csv", sep=";")
mapping = pd.read_csv("submissions_archive/task_competence_mapping.csv", sep=";")
micro = pd.read_csv("submissions_archive/micromodules_competencies.csv", sep=";")

In [6]:
task_map = pd.read_csv("tasks_latent_space_clusters.csv")
task_map.head()

,task_id,task_number,codifier,dataset_source,text_clean,cluster,x_umap,y_umap,x_tsne,y_tsne
0,40B442,2,['7.5 Координаты и векторы'],fipi,Даны векторы overrightarrow{a} (25; 0) и overr...,4,-5.752996,-7.214250,-24.808470,8.073481
1,4CBD4E,6,"['2.1 Целые и дробно-рациональные уравнения', ...",fipi,Найдите корень уравнения (1 7) x + 4 = 49.,4,-3.448695,7.386013,2.810702,32.918064
2,FDA042,13,"['2.3 Тригонометрические уравнения', '2.4 Пока...",fipi,а) Решите уравнение 9 sin 2 x - 3 2 2 sin x 11...,10,-13.362803,-4.066293,-44.377724,26.449654
3,0A394E,2,['7.5 Координаты и векторы'],fipi,Даны векторы a → ( - 13; 4) и b → ( - 6; 1). Н...,4,-5.724255,-7.242629,-26.733840,9.473522
4,75244F,4,['6.2 Вероятность'],fipi,Фабрика выпускает сумки. В среднем 4 сумки из ...,13,-2.721873,-3.838118,45.566364,-22.050450


In [7]:
user_number = 13

trajectory = (
    tasks[tasks["user_number"] == user_number]
    .sort_values("submission_time")
    .reset_index(drop=True)
)

trajectory.head()

,user_id,user_number,submission_id,practice_submission_id,task_id,task_name,task_type,max_score,node_id,node_name,course_id,task_number,status,score,autochecked,submission_time,is_correct
0,7ba7e77a-3fa6-48e7-94a6-42ffd0a3363e,13,4501b175-5209-4a34-9248-9f9163e908b2,828503ac-b2fb-4159-8046-96155a3893f6,a580f2d9-4fc1-4a83-be85-8420d3f458a1,№1 - ФИПИ - 40B442,SHORT_TEXT,1.0,25570988-2dad-48c6-9974-af6a4a36c3f6,Номер 2 ЕГЭ,bc1d4037-e5be-4ea4-94d8-e812a1343379,2,REJECTED,0,True,2026-02-18 19:41:18.709332+00:00,False
1,7ba7e77a-3fa6-48e7-94a6-42ffd0a3363e,13,f6a0a789-83f8-4a57-91bf-ecad94b24522,828503ac-b2fb-4159-8046-96155a3893f6,1160c2b7-f837-4e34-a6c5-4565645e07c7,№3 - ФИПИ - cD7468,SHORT_TEXT,1.0,cb8d3596-0857-4bc5-addd-96612fd11596,Номер 3 ЕГЭ,d00a64e6-a938-4c69-8b91-bc754bdcb80e,3,REJECTED,0,True,2026-02-18 19:41:18.709332+00:00,False
2,7ba7e77a-3fa6-48e7-94a6-42ffd0a3363e,13,6259dd4f-a6d6-4cef-98df-6bc92fbca3e6,828503ac-b2fb-4159-8046-96155a3893f6,1ce18833-1a76-4062-9c1d-8e883de9e030,№11 - ФИПИ - 95F2C4,SHORT_TEXT,1.0,11bcc943-1b0c-4a91-8b8d-e3c4c6da81c6,15. Классическая вероятность для 3 испытаний: ...,0be1191d-2a75-4408-90c6-e45ea709e43e,4,REJECTED,0,True,2026-02-18 19:41:18.709332+00:00,False
3,7ba7e77a-3fa6-48e7-94a6-42ffd0a3363e,13,77324bcc-3dfa-43b1-9767-8936d3948938,828503ac-b2fb-4159-8046-96155a3893f6,330c1b44-4496-4bd1-9445-64003a4322f1,№41 - ФИПИ - 0F9157,SHORT_TEXT,1.0,b25788c9-58ae-4659-9527-b8e1e748b274,Номер 5 ЕГЭ,5bc61ff7-6447-44a1-b441-a768e6902f9b,5,REJECTED,0,True,2026-02-18 19:41:18.709332+00:00,False
4,7ba7e77a-3fa6-48e7-94a6-42ffd0a3363e,13,9c5d9a68-0aa6-4208-a4f9-c2c1705a1054,828503ac-b2fb-4159-8046-96155a3893f6,b5fa8a87-5c94-470f-b5b5-1e7cedb3be9a,№22 - ФИПИ - D14D52,SHORT_TEXT,1.0,538dbc01-658a-4f92-9cb8-0d004586d49c,15. Решение показательного уравнения с приведе...,830dd404-3b3f-4a21-af65-5ddbf150b6ed,6,REJECTED,0,True,2026-02-18 19:41:18.709332+00:00,False


In [17]:
import re

def extract_fipi_id(name):
    if pd.isna(name):
        return None
    match = re.search(r'([A-F0-9]{6})$', name)
    return match.group(1) if match else None

trajectory["task_id"] = trajectory["task_name"].apply(extract_fipi_id)

In [18]:
trajectory_small = trajectory[[
    "user_number",
    "submission_time",
    "task_id",
    "task_name",
    "task_number",
    "node_name",
    "is_correct",
    "score",
    "max_score"
]].copy()

In [19]:
traj_umap = trajectory_small.merge(
    task_map[[
        "task_id",
        "x_umap",
        "y_umap",
        "x_tsne",
        "y_tsne",
        "cluster",
        "text_clean",
        "codifier"
    ]],
    on="task_id",
    how="left"
)

In [20]:
print(traj_umap.shape)
print(traj_umap[["task_id", "x_umap", "y_umap"]].isna().sum())

(420, 16)
task_id    49
x_umap     49
y_umap     49
dtype: int64


In [21]:
traj_umap = traj_umap.dropna(subset=["x_umap", "y_umap"]).copy()
traj_umap = traj_umap.sort_values("submission_time").reset_index(drop=True)

print(traj_umap.shape)
traj_umap.head()

(371, 16)


,user_number,submission_time,task_id,task_name,task_number,node_name,is_correct,score,max_score,x_umap,y_umap,x_tsne,y_tsne,cluster,text_clean,codifier
0,13,2026-02-18 19:41:18.709332+00:00,40B442,№1 - ФИПИ - 40B442,2,Номер 2 ЕГЭ,False,0,1.0,-5.752996,-7.214250,-24.808470,8.073481,4.0,Даны векторы overrightarrow{a} (25; 0) и overr...,['7.5 Координаты и векторы']
1,13,2026-02-18 19:41:18.709332+00:00,95F2C4,№11 - ФИПИ - 95F2C4,4,15. Классическая вероятность для 3 испытаний: ...,False,0,1.0,-9.316226,7.380847,37.168590,4.750828,6.0,Перед началом футбольного матча судья бросает ...,['6.2 Вероятность']
2,13,2026-02-18 19:41:18.709332+00:00,0F9157,№41 - ФИПИ - 0F9157,5,Номер 5 ЕГЭ,False,0,1.0,-9.053912,6.660183,38.056232,-9.249352,13.0,"В коробке 12 синих, 6 красных и 7 зелёных флом...",['6.2 Вероятность']
3,13,2026-02-18 19:41:18.709332+00:00,D14D52,№22 - ФИПИ - D14D52,6,15. Решение показательного уравнения с приведе...,False,0,1.0,-3.608179,7.544604,1.751685,28.946163,4.0,Найдите корень уравнения 9 - 2 - x = 81.,['2.4 Показательные и логарифмические уравнения']
4,13,2026-02-18 19:41:18.709332+00:00,68CF2D,№30 - ФИПИ - 68CF2D,7,Номер 7 ЕГЭ,False,0,1.0,3.247174,20.415087,4.427987,44.291203,4.0,Найдите значение выражения 36 3 ⋅ 36 5 36 30.,['1.3 Арифметический корень натуральной степен...


In [22]:
# Только траектория пользователя

fig = px.line(
    traj_umap,
    x="x_umap",
    y="y_umap",
    markers=True,
    hover_data=[
        "submission_time",
        "task_id",
        "task_name",
        "task_number",
        "node_name",
        "cluster",
        "is_correct"
    ],
    title=f"Траектория пользователя {user_number} на UMAP-карте задач"
)

fig.show()

In [23]:
# Первые 50 шагов
fig = px.line(
    traj_umap.head(50),
    x="x_umap",
    y="y_umap",
    markers=True,
    hover_data=[
        "submission_time",
        "task_id",
        "task_name",
        "task_number",
        "node_name",
        "cluster",
        "is_correct"
    ],
    title=f"Первые 50 шагов пользователя {user_number} на UMAP-карте"
)

fig.show()

In [32]:
traj_plot = traj_umap.head(5).copy()

In [33]:
# общая карта задач + путь пользователя
import plotly.graph_objects as go

fig = go.Figure()

# Все задачи как фон
fig.add_trace(go.Scatter(
    x=task_map["x_umap"],
    y=task_map["y_umap"],
    mode="markers",
    marker=dict(size=5, opacity=0.25),
    name="Все задачи",
    text=task_map["task_id"],
    hoverinfo="skip"
))
# траектории пользователя
fig.add_trace(go.Scatter(
    x=traj_plot["x_umap"],
    y=traj_plot["y_umap"],
    mode="lines+markers",
    marker=dict(size=7),
    line=dict(width=2),
    name=f"Путь пользователя {user_number}",
    text=traj_plot["task_name"],
    hovertemplate=
        "<b>%{text}</b><br>" +
        "task_id: %{customdata[0]}<br>" +
        "time: %{customdata[1]}<br>" +
        "cluster: %{customdata[2]}<br>" +
        "correct: %{customdata[3]}<extra></extra>",
    customdata=np.stack([
        traj_plot["task_id"],
        traj_plot["submission_time"].astype(str),
        traj_plot["cluster"].astype(str),
        traj_plot["is_correct"].astype(str)
    ], axis=1)
))

fig.update_layout(
    title=f"Траектория пользователя {user_number} на карте задач (UMAP)",
    xaxis_title="x_umap",
    yaxis_title="y_umap",
    width=1000,
    height=700
)

fig.show()